In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. SETUP & STOCHASTIC DEMAND GENERATION
# ==========================================
np.random.seed(42) # For reproducibility

TIME_STEPS = 96 # 15-minute intervals over 24 hours
time_hours = np.linspace(0, 24, TIME_STEPS)
MAX_POWER = 250 # MW for Phase 3

# Base Diurnal Curve (Normal Day)
mean_u = 0.4 + 0.3 * np.sin(np.pi * (time_hours - 6) / 12) + 0.1 * np.sin(np.pi * (time_hours - 14) / 6)

# Autocorrelated Stochastic Noise
noise = np.random.normal(0, 0.05, TIME_STEPS)
smoothed_noise = pd.Series(noise).rolling(window=4, min_periods=1).mean().values
u_sim = np.clip(mean_u + smoothed_noise, 0.1, 1.0)

# Convert Utilization to MW (factoring in Idle Load and PUE)
PUE = 1.2
IDLE_FRACTION = 0.20
max_it_power = MAX_POWER / PUE
base_it_power = max_it_power * IDLE_FRACTION

demand_mw = (base_it_power + (max_it_power - base_it_power) * u_sim) * PUE
demand_mw = np.clip(demand_mw, 0, MAX_POWER) # Hard cap at facility limit

# Helper: Solar PPA Generation Curve (Daylight hours only)
solar_curve = np.zeros(TIME_STEPS)
daylight = (time_hours >= 7) & (time_hours <= 19)
solar_curve[daylight] = np.sin(np.pi * (time_hours[daylight] - 7) / 12) # Peaks at 1:00 PM

# Helper: Evening Peak Index for BESS discharge (6:00 PM to 10:00 PM)
evening = (time_hours >= 18) & (time_hours <= 22)

# ==========================================
# 2. DISPATCH LOGIC FOR SCENARIOS
# ==========================================
dispatch_data = {}

# Baseline: 100% Grid
dispatch_data['Baseline (100% Grid)'] = {
    'Grid': demand_mw, 'RICE': np.zeros(TIME_STEPS), 'Solar PPA': np.zeros(TIME_STEPS),
    'Geothermal': np.zeros(TIME_STEPS), 'SMR': np.zeros(TIME_STEPS),
    'Microturbines': np.zeros(TIME_STEPS), 'BESS (Discharge)': np.zeros(TIME_STEPS)
}

# Scenario 1: Grid (100 MW anchor) + RICE + BESS
s1_grid = np.minimum(demand_mw, 100)
s1_remainder = demand_mw - s1_grid
s1_bess = np.where(s1_remainder > 130, s1_remainder - 130, 0) # BESS shaves peaks above 230MW
s1_rice = s1_remainder - s1_bess
dispatch_data['S1: Grid + RICE + BESS'] = {
    'Grid': s1_grid, 'RICE': s1_rice, 'BESS (Discharge)': s1_bess,
    'Solar PPA': np.zeros(TIME_STEPS), 'Geothermal': np.zeros(TIME_STEPS),
    'SMR': np.zeros(TIME_STEPS), 'Microturbines': np.zeros(TIME_STEPS)
}

# Scenario 2: Solar PPA + RICE + BESS + Grid Backup (10 MW max)
s2_solar_used = np.minimum(demand_mw, solar_curve * 120) # 120 MW Solar peak
s2_grid = np.minimum(demand_mw - s2_solar_used, 10)
s2_remainder = demand_mw - s2_solar_used - s2_grid
s2_bess = np.zeros(TIME_STEPS)
s2_bess[evening] = np.minimum(s2_remainder[evening], 40) # 40 MW BESS evening discharge
s2_rice = s2_remainder - s2_bess
dispatch_data['S2: Solar PPA + RICE + BESS + Grid Backup'] = {
    'Grid': s2_grid, 'RICE': s2_rice, 'BESS (Discharge)': s2_bess, 'Solar PPA': s2_solar_used,
    'Geothermal': np.zeros(TIME_STEPS), 'SMR': np.zeros(TIME_STEPS), 'Microturbines': np.zeros(TIME_STEPS)
}

# Scenario 3: 100% Geothermal Transition
s3_geo = demand_mw
dispatch_data['S3: 100% Geothermal Baseload'] = {
    'Grid': np.zeros(TIME_STEPS), 'RICE': np.zeros(TIME_STEPS), 'BESS (Discharge)': np.zeros(TIME_STEPS),
    'Solar PPA': np.zeros(TIME_STEPS), 'Geothermal': s3_geo, 'SMR': np.zeros(TIME_STEPS), 'Microturbines': np.zeros(TIME_STEPS)
}

# Scenario 4: Geothermal (120 MW) + Solar PPA (80 MW) + BESS + Grid
s4_geo = np.minimum(demand_mw, 120)
s4_remainder_1 = demand_mw - s4_geo
s4_solar_used = np.minimum(s4_remainder_1, solar_curve * 80)
s4_remainder_2 = s4_remainder_1 - s4_solar_used
s4_bess = np.zeros(TIME_STEPS)
s4_bess[evening] = np.minimum(s4_remainder_2[evening], 30)
s4_grid = s4_remainder_2 - s4_bess
dispatch_data['S4: Geothermal + Solar PPA + BESS + Grid'] = {
    'Grid': s4_grid, 'RICE': np.zeros(TIME_STEPS), 'BESS (Discharge)': s4_bess, 'Solar PPA': s4_solar_used,
    'Geothermal': s4_geo, 'SMR': np.zeros(TIME_STEPS), 'Microturbines': np.zeros(TIME_STEPS)
}

# Scenario 5: SMR (180 MW) + Solar PPA (50 MW) + Grid
s5_smr = np.minimum(demand_mw, 180)
s5_remainder = demand_mw - s5_smr
s5_solar_used = np.minimum(s5_remainder, solar_curve * 50)
s5_grid = s5_remainder - s5_solar_used
dispatch_data['S5: SMR + Solar PPA + Grid'] = {
    'Grid': s5_grid, 'RICE': np.zeros(TIME_STEPS), 'BESS (Discharge)': np.zeros(TIME_STEPS), 'Solar PPA': s5_solar_used,
    'Geothermal': np.zeros(TIME_STEPS), 'SMR': s5_smr, 'Microturbines': np.zeros(TIME_STEPS)
}

# Scenario 6: Geothermal (100 MW) + Grid (50 MW) + Microturbines + BESS
s6_geo = np.minimum(demand_mw, 100)
s6_grid = np.minimum(demand_mw - s6_geo, 50)
s6_remainder = demand_mw - s6_geo - s6_grid
s6_bess = np.zeros(TIME_STEPS)
s6_bess[evening] = np.minimum(s6_remainder[evening], 20)
s6_micro = s6_remainder - s6_bess
dispatch_data['S6: Geothermal + Grid + Microturbines + BESS'] = {
    'Grid': s6_grid, 'RICE': np.zeros(TIME_STEPS), 'BESS (Discharge)': s6_bess, 'Solar PPA': np.zeros(TIME_STEPS),
    'Geothermal': s6_geo, 'SMR': np.zeros(TIME_STEPS), 'Microturbines': s6_micro
}

# ==========================================
# 3. PLOTTING THE STACKED AREA GRAPHS
# ==========================================
fig, axes = plt.subplots(7, 1, figsize=(14, 25), sharex=True)
plt.subplots_adjust(hspace=0.4)

# Define uniform colors for the stacked areas
colors = {
    'Grid': '#aaaaaa',
    'Geothermal': '#8b4513',
    'SMR': '#4b0082',
    'Solar PPA': '#ffcc00',
    'RICE': '#ff6600',
    'Microturbines': '#d2691e',
    'BESS (Discharge)': '#00cc66'
}

sources = ['Geothermal', 'SMR', 'Grid', 'Solar PPA', 'RICE', 'Microturbines', 'BESS (Discharge)']

for ax, (title, data) in zip(axes, dispatch_data.items()):
    # Extract the arrays in the specific stacking order
    y_stack = [data[source] for source in sources]

    # Create the stacked plot
    ax.stackplot(time_hours, y_stack, labels=sources, colors=[colors[s] for s in sources], alpha=0.85)

    # Overlay the total demand curve as a strict black line
    ax.plot(time_hours, demand_mw, color='black', linewidth=2, label='Total Demand')

    ax.set_title(f'Phase 3 (250 MW) Dispatch Profile - {title}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Power (MW)')
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 260)
    ax.grid(True, linestyle=':', alpha=0.6)

    # Place the legend only on the first plot to avoid clutter
    if title == 'Baseline (100% Grid)':
        ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1.05), fontsize=10)

axes[-1].set_xlabel('Hour of Day (CST)')

# Save out the high-resolution file
plt.savefig('generation_dispatch_phase3.png', dpi=300, bbox_inches='tight')
plt.show()

SyntaxError: unterminated string literal (detected at line 46) (3228210486.py, line 46)